# Thu nap Vintern-1B tren transformers cu

Vintern **chua tung nap duoc lan nao** -- moi so 100% trong bao cao cu deu la mock.

Notebook nay chi chay 3 anh (~5 phut) de tra loi mot cau: nap duoc khong, va neu
khong thi hong o dau. Chay du 355 anh chi lam sau khi cai nay xanh.

`InternVLChatModel` thieu `all_tied_weights_keys` ma transformers 4.5x+ doi hoi,
nen phai cai ban cu hon -- KHONG dung mocs `>=4.51,<5` cua Qwen.


In [ ]:
import os

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch

print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHONG CO')
assert torch.cuda.is_available(), 'Chua bat GPU'


In [ ]:
# Moc CU HON Qwen: Vintern can transformers truoc khi all_tied_weights_keys
# thanh bat buoc. Ghim ca timm/einops vi InternVL goi toi qua trust_remote_code.
!pip install -q "transformers>=4.37,<4.50" accelerate bitsandbytes timm einops sentencepiece
import transformers
print('transformers:', transformers.__version__)
assert transformers.__version__ < '4.50', f'pip keo nham ban {transformers.__version__}'


In [ ]:
import subprocess, sys
from pathlib import Path

REPO = 'https://github.com/lolizabrett-byte/Multimodal-Agentic-Retrieval-Engine.git'
NHANH = 'research/vlm-prompting'
DICH = Path('/kaggle/working/repo')

if not DICH.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', NHANH, REPO, str(DICH)],
                   check=True)

PKG = DICH / 'system1' / 'research' / 'vlm_prompting'
assert PKG.exists(), f'Khong thay code tai {PKG}'
sys.path.insert(0, str(PKG))
print('Code tai:', PKG)

hash_code = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                           cwd=DICH, capture_output=True, text=True).stdout.strip()
print('Commit:', hash_code)


In [ ]:
# TANG 1 -- nap model. In nguyen van loi neu hong: bao cao hien chi co
# chan doan gian tiep, phien nay la co hoi lay bang chung truc tiep.
import traceback

nap_duoc = False
try:
    from vlm.model_loader import load_model
    model, processor, spec = load_model('vintern-1b')
    nap_duoc = True
    print('TANG 1 OK -- nap duoc:', spec.hf_id)
    print('   kieu model:', type(model).__name__)
except Exception:
    print('TANG 1 HONG -- khong nap duoc. Nguyen van:')
    traceback.print_exc()


In [ ]:
# TANG 2 -- chay infer() that. adapters.py khong co xu ly rieng cho InternVL,
# no di qua TransformersAdapter dung .generate(). Neu InternVL doi .chat() rieng
# thi tang nay hong du tang 1 xanh.
ANH_DIR = next(Path('/kaggle/input').glob('**/images'), None)
anh_thu = sorted(ANH_DIR.glob('*.jpg'))[:3]
print('Anh thu:', [a.name for a in anh_thu])

ket_qua_thu = []
if not nap_duoc:
    print('Bo qua tang 2 -- tang 1 da hong')
else:
    from vlm.generate import generate_json
    for a in anh_thu:
        try:
            kq = generate_json(a, model_key='vintern-1b', backend='transformers')
            ket_qua_thu.append(kq)
            print(f'  {a.name}: OK | backend={kq.get("_backend")} '
                  f'| {kq.get("_latency_sec")}s')
            print(f'     caption: {str(kq.get("caption_chi_tiet"))[:110]}')
        except Exception as loi:
            print(f'  {a.name}: HONG -- {type(loi).__name__}: {str(loi)[:300]}')


In [ ]:
# TANG 3 -- ket luan. mock hoac latency 0.0 nghia la VAN chua nap duoc that,
# du cac cell tren khong nem loi.
print('=' * 60)
if not ket_qua_thu:
    print('KET LUAN: Vintern KHONG chay duoc. Doc loi nguyen van o cell tren.')
else:
    backends = {k.get('_backend') for k in ket_qua_thu}
    lat = [float(k.get('_latency_sec') or 0) for k in ket_qua_thu]
    that = 'mock' not in str(backends) and all(x > 0 for x in lat)
    print(f'backend: {backends} | latency: {lat}')
    if that:
        print('KET LUAN: Vintern CHAY THAT. Chay tiep 355 anh duoc.')
    else:
        print('KET LUAN: van la MOCK -- so khong dung duoc. Khong chay 355 anh.')
